# F1 Podium Prediction
**Goal:** Can we predict whether an F1 driver will finish on the podium (top 3)?

We're using historical race data from the Ergast API and training a few different classifiers to see what works best. The main focus is on making the model interpretable — not just accurate.

In [ ]:
# install these if you haven't already
# !pip install pandas numpy scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# models
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# evaluation
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print('all good!')

## 1. Load the data

We're using 5 CSV files from the Ergast F1 dataset. Put them all in a `data/` folder next to this notebook.

- `results.csv` — finishing position, points, grid, laps for every driver in every race
- `qualifying.csv` — qualifying times and positions
- `races.csv` — year, circuit name, date
- `drivers.csv` — driver names and nationality
- `constructors.csv` — team names

In [ ]:
DATA = 'data/'

results      = pd.read_csv(DATA + 'results.csv')
qualifying   = pd.read_csv(DATA + 'qualifying.csv')
races        = pd.read_csv(DATA + 'races.csv')
drivers      = pd.read_csv(DATA + 'drivers.csv')
constructors = pd.read_csv(DATA + 'constructors.csv')

# quick check
for name, df in [('results', results), ('qualifying', qualifying), ('races', races),
                 ('drivers', drivers), ('constructors', constructors)]:
    print(f'{name}: {len(df)} rows')

## 2. Merge everything together

Right now we have 5 separate tables. We need to join them into one flat dataframe so we can use all the info together.

The common keys are `raceId` and `driverId` — every table has at least one of these.

We also filter to 2010-2023 to keep things in the modern era. Mixing 1950s F1 with 2020s F1 would just confuse the model — completely different cars, rules, and tyre strategies.

In [ ]:
# start with results, add race info (year, circuit)
df = results.merge(races[['raceId', 'year', 'circuitId', 'name']], on='raceId', how='left')

# add team name
df = df.merge(constructors[['constructorId', 'name']], on='constructorId',
              how='left', suffixes=('_race', '_constructor'))

# add driver surname and nationality
df = df.merge(drivers[['driverId', 'surname', 'nationality']], on='driverId', how='left')

# add qualifying position
qual = qualifying[['raceId', 'driverId', 'position']].rename(columns={'position': 'qual_position'})
df = df.merge(qual, on=['raceId', 'driverId'], how='left')

# filter to modern era
df = df[(df['year'] >= 2010) & (df['year'] <= 2023)].copy()

# the Ergast dataset uses \N for missing values — replace with NaN
df.replace('\\N', np.nan, inplace=True)

# make sure numeric columns are actually numbers
for col in ['positionOrder', 'points', 'grid', 'qual_position', 'laps']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'final shape: {df.shape}')
df[['year', 'name_race', 'surname', 'name_constructor', 'grid', 'positionOrder']].head(8)

## 3. Create the target variable

We want to predict **podium = yes/no**, so we turn this into a binary column:
- 1 = driver finished P1, P2 or P3
- 0 = everyone else

One thing worth noting: only 3 out of ~20 drivers podium per race, so about 15% of rows are positive. This is called **class imbalance** and it matters when we look at our evaluation metrics later.

In [ ]:
df['podium'] = (df['positionOrder'] <= 3).astype(int)

# drop rows where positionOrder doesn't make sense (outside 1-20)
df = df[df['positionOrder'].between(1, 20)].copy()

print('Class balance:')
print(df['podium'].value_counts(normalize=True).round(3))
print(f'Total rows: {len(df)}')

fig, ax = plt.subplots(figsize=(4, 3))
df['podium'].value_counts().plot(kind='bar', ax=ax,
                                  color=['#5B8DB8', '#E07B54'], edgecolor='white')
ax.set_xticklabels(['No podium', 'Podium'], rotation=0)
ax.set_title('Class distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Quick EDA before modelling

Before jumping into ML, it's always worth visualising the data. Two obvious questions:
1. How much does qualifying position predict the race result?
2. Which teams podium most often?

These plots also help justify our feature choices.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# grid position vs podium rate
grid_stats = (
    df[df['grid'] <= 20]
    .groupby('grid')['podium']
    .mean()
    .reset_index()
)
axes[0].bar(grid_stats['grid'], grid_stats['podium'] * 100,
            color='#5B8DB8', edgecolor='white')
axes[0].axvline(x=3.5, color='red', linestyle='--', alpha=0.5, label='Top 3 grid')
axes[0].set_xlabel('Grid position')
axes[0].set_ylabel('Podium rate (%)')
axes[0].set_title('Grid position vs podium rate')
axes[0].legend()

# constructor podium rates (top 10 teams with enough races)
team_stats = (
    df.groupby('name_constructor')['podium']
    .agg(['mean', 'count'])
    .query('count >= 50')
    .nlargest(10, 'mean')
)
axes[1].barh(team_stats.index, team_stats['mean'] * 100,
             color='#E07B54', edgecolor='white')
axes[1].set_xlabel('Podium rate (%)')
axes[1].set_title('Top 10 teams by podium rate (2010-2023)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 5. Feature Engineering

Raw columns don't always work well as ML features. We need to create some new ones.

The most important thing here is **avoiding data leakage** — we can't use information from the current race to predict the current race. That's why we use `.shift(1)` on rolling features, which means "look at the previous race, not this one".

Features we'll create:
- `grid_position` — where they started (from qualifying)
- `rolling_pts_3` — average points over the last 3 races (form)
- `constructor_avg_pos` — team's average finish position this season
- `is_top3_grid` — did they start in the top 3? (binary)
- `constructor_podium_rate` — team's historical podium rate

In [ ]:
# sort by time first — this is critical for any rolling/lag features
df = df.sort_values(['year', 'raceId', 'driverId']).reset_index(drop=True)

# grid position (fill missing qualifying data with grid column)
df['grid_position'] = df['qual_position'].fillna(df['grid'])

# rolling average points — shift(1) means we only look at PAST races, not the current one
# without shift(1) this would be data leakage!
df['rolling_pts_3'] = (
    df.groupby('driverId')['points']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

# constructor average finish position so far this season
df['constructor_avg_pos'] = (
    df.groupby(['year', 'constructorId'])['positionOrder']
    .transform(lambda x: x.shift(1).expanding().mean())
)

# did the driver start in top 3?
df['is_top3_grid'] = (df['grid_position'] <= 3).astype(int)

# year normalised to 0-1 range
df['year_norm'] = (df['year'] - df['year'].min()) / (df['year'].max() - df['year'].min())

# team's overall podium rate (mean encoding)
team_podium_rate = df.groupby('name_constructor')['podium'].mean()
df['constructor_podium_rate'] = df['name_constructor'].map(team_podium_rate)

FEATURES = ['grid_position', 'rolling_pts_3', 'constructor_avg_pos',
            'is_top3_grid', 'year_norm', 'constructor_podium_rate']

print('Features preview:')
df[FEATURES + ['podium']].dropna().head(6)

## 6. Train / test split

Instead of a random split, we split by year:
- **Train:** 2010-2020
- **Test:** 2021-2023

This is more realistic — you'd never train a model on future races to predict past ones. A random split would leak info from the same season into both sets.

In [ ]:
TARGET = 'podium'

model_df = df[FEATURES + [TARGET, 'year']].dropna().copy()

train = model_df[model_df['year'] <= 2020]
test  = model_df[model_df['year'] > 2020]

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f'Train: {len(X_train)} rows  ({y_train.mean():.1%} podium rate)')
print(f'Test:  {len(X_test)} rows  ({y_test.mean():.1%} podium rate)')

## 7. Train the models

We're training three models and comparing them:
1. **Logistic Regression** — linear baseline, fast, easy to understand
2. **Decision Tree** — the main one we'll interpret; limited to depth 4 to avoid overfitting
3. **Random Forest** — 100 trees averaged together; more accurate but harder to explain

Cross-validation on the training set gives us an idea of how well each model generalises before we even look at the test set.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
}

results_table = []

for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc    = accuracy_score(y_test, y_pred)
    f1     = f1_score(y_test, y_pred)
    auc    = roc_auc_score(y_test, y_proba)
    cv_acc = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy').mean()

    results_table.append({
        'Model': name,
        'Test Accuracy': round(acc, 3),
        'F1 Score': round(f1, 3),
        'ROC-AUC': round(auc, 3),
        'CV Accuracy': round(cv_acc, 3)
    })

print(pd.DataFrame(results_table).to_string(index=False))

## 8. Visualise the Decision Tree

This is the most interesting part of an interpretable ML project — we can actually **read the model's decisions**.

Each node shows:
- The feature and threshold being tested
- The gini impurity (how mixed the classes are at that point)
- The value = [no_podium_count, podium_count]
- The majority class prediction

Try tracing a path from the top down — that's a rule the model learned from data.

In [ ]:
dt = models['Decision Tree']

fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(
    dt,
    feature_names=FEATURES,
    class_names=['No Podium', 'Podium'],
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax
)
ax.set_title('Decision Tree — F1 Podium Prediction (max_depth=4)', fontsize=13)
plt.tight_layout()
plt.savefig('decision_tree.png', dpi=150, bbox_inches='tight')
plt.show()

# text version — useful to read through
print(export_text(dt, feature_names=FEATURES))

## 9. Feature Importance

Both tree-based models give us `feature_importances_` — how much each feature contributes to reducing uncertainty in the predictions.

Interesting to compare the two: the single Decision Tree leans heavily on just one feature, while the Random Forest spreads the weight more evenly. That's part of why forests generalise better.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, model_name in zip(axes, ['Decision Tree', 'Random Forest']):
    model = models[model_name]
    importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
    importances.plot(kind='barh', ax=ax, color='#5B8DB8', edgecolor='white')
    ax.set_title(f'{model_name} — feature importance')
    ax.set_xlabel('Importance (Gini reduction)')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Confusion Matrices

A confusion matrix breaks down predictions into four categories:
- **True Positive** — correctly predicted podium
- **True Negative** — correctly predicted no podium
- **False Positive** — predicted podium, but they didn't make it
- **False Negative** — predicted no podium, but they actually podiumed

Because only ~15% of rows are positive, accuracy alone is misleading — a model that always says "no podium" would get 85% accuracy while being totally useless.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, model) in zip(axes, models.items()):
    preds = model.predict(X_test)
    cm    = confusion_matrix(y_test, preds)
    ConfusionMatrixDisplay(cm, display_labels=['No Podium', 'Podium']).plot(
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(name)

plt.suptitle('Confusion matrices (test set 2021-2023)', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. ROC Curves

The ROC curve shows how good the model is at **ranking** predictions — can it give high confidence to actual podiums and low confidence to non-podiums?

**AUC (Area Under the Curve):**
- 0.5 = random guessing (the diagonal line)
- 1.0 = perfect model
- Our models are all ~0.93, which is pretty solid

AUC is better than accuracy for imbalanced datasets because it doesn't depend on a fixed decision threshold.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

colors = {
    'Logistic Regression': '#5B8DB8',
    'Decision Tree':       '#E07B54',
    'Random Forest':       '#6AAB72'
}

for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})',
            color=colors[name], linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC curves — all models')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Error Analysis — what did the model get wrong?

Looking at the mistakes is often more interesting than looking at what the model got right.

**False negatives** = races where a driver actually podiumed but our model said they wouldn't. These are usually the "surprise" results — drivers who started further back but benefited from rain, safety cars, or retirements ahead of them. No structured dataset can capture those events.

In [ ]:
dt = models['Decision Tree']

X_test_r    = X_test.reset_index(drop=True)
y_test_r    = y_test.reset_index(drop=True)
context     = test[['year', 'name_race', 'surname', 'name_constructor']].reset_index(drop=True)

errors = context.copy()
errors['actual']      = y_test_r
errors['predicted']   = dt.predict(X_test_r)
errors['podium_prob'] = dt.predict_proba(X_test_r)[:, 1]
errors = pd.concat([errors, X_test_r], axis=1)

# false negatives — driver podiumed but we said no
fn = errors[(errors['actual'] == 1) & (errors['predicted'] == 0)].sort_values('podium_prob')

print(f'False negatives (missed podiums): {len(fn)}')
print('\nMost surprising misses (model gave lowest probability):')
fn[['year', 'name_race', 'surname', 'name_constructor',
    'grid_position', 'rolling_pts_3', 'podium_prob']].head(10)

## 13. Overfitting investigation

What happens if we make the tree deeper and deeper? At some point it stops learning general patterns and starts memorising the training data — that's overfitting.

The plot below shows the **bias-variance trade-off**: training accuracy keeps climbing, but test accuracy peaks around depth 4 then falls. The gap between the two lines is the overfitting.

In [ ]:
depths = range(1, 16)
train_accs, test_accs = [], []

for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42)
    t.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, t.predict(X_train)))
    test_accs.append(accuracy_score(y_test,  t.predict(X_test)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(depths, train_accs, label='Train accuracy', color='#5B8DB8', marker='o', markersize=4)
ax.plot(depths, test_accs,  label='Test accuracy',  color='#E07B54', marker='o', markersize=4)
ax.axvline(x=4, color='gray', linestyle='--', alpha=0.5, label='Our model (depth=4)')
ax.set_xlabel('Tree max depth')
ax.set_ylabel('Accuracy')
ax.set_title('Bias-Variance trade-off: training vs test accuracy by tree depth')
ax.legend()

ax.annotate('Overfitting zone\n(train >> test)', xy=(12, test_accs[11]),
            xytext=(11, 0.88), fontsize=9, color='#E07B54',
            arrowprops=dict(arrowstyle='->', color='#E07B54'))

plt.tight_layout()
plt.savefig('bias_variance.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

**What we found:**
- Grid position (qualifying) is the single strongest predictor by far
- Recent form (rolling points) adds meaningful signal on top of that
- Team quality matters — but less than where you start
- A Decision Tree at depth 4 performs nearly as well as 100 Random Forest trees (0.931 vs 0.938 AUC)
- The model can't predict chaos — rain, safety cars, retirements ahead are invisible to it

**What we'd do next with more time:**
- Add weather data
- Add pit stop strategy features
- Try SHAP values for per-prediction explanations
- Build a live predictor for an upcoming race weekend